# 04. Interactive Plotly Visualizations
## 📚 Learning Objectives

By completing this notebook, you will:
- Create interactive Plotly charts
- Use hover, zoom, and other interactivity
- Export or embed interactive visualizations

## 🔗 Where this fits

**Builds on:** Course 05 — Unit 3, lessons 01-03 (static charts) — the same chart types, now with hover, zoom and selection.

**Used later in:** Course 05 — Unit 3, lesson 05, which builds dashboards and exports self-contained HTML.

---


**All concepts are explained in the code comments below - you can learn everything from this notebook alone!**

---

## 🔗 Solving the Problem from Example 3

**Remember the dead end from Example 3?**
- We learned Seaborn for beautiful statistical visualizations
- But Seaborn creates static plots - can't zoom, pan, or explore interactively
- We needed interactive plots for data exploration and dashboards

**This notebook solves that problem!**
- We'll learn **Plotly** - interactive plotting library
- We'll create plots that can be **zoomed, panned, and explored**
- We'll build **interactive dashboards** for presentations and sharing

**This solves the interactivity problem from Example 3!**

---

## The Story: From Photos to Interactive Maps

Imagine you're showing a location. A photo shows one view, but an interactive map 
lets people zoom, pan, and explore. **After** seeing interactivity, people understand better!

Same with visualization: Static plots show one view, but interactive plots let users 
zoom, pan, hover for details, and explore. **After** experiencing interactivity, data 
communication is much more effective!

---

## Why Interactive Visualizations Matter

Interactive visualizations are powerful because:
- **Exploration**: Users can zoom, pan, and explore data themselves
- **Details**: Hover to see exact values and details
- **Dashboards**: Perfect for live dashboards and presentations
- **Engagement**: More engaging than static plots
- **Sharing**: Great for web-based reports and presentations

## Learning Objectives
1. Create interactive plots with Plotly
2. Understand when to use interactive vs static plots
3. Create interactive scatter, line, and bar plots
4. Add interactivity features (hover, zoom, pan)
5. Export interactive plots to HTML for sharing


## 📥 Inputs & 📤 Outputs

**Inputs:** What we use in this notebook

- **Real data:** `montgomery_911_calls.csv` — every 911 emergency call dispatched in
  Montgomery County, Pennsylvania from December 2015 to July 2020 (663,522 real calls).
  Columns: latitude/longitude of the incident, dispatch timestamp, township, ZIP code,
  and a `title` such as `EMS: CARDIAC EMERGENCY` or `Traffic: VEHICLE ACCIDENT -`.
- plotly, pandas

**Outputs:** What you'll see when you run the cells

- Interactive charts of the real call log (map-like scatter, monthly trend, call-type bars)
- Self-contained HTML files you can open in any browser

---


In [1]:
# WHAT: Import Plotly's two APIs - express (quick) and graph_objects (full control).
# WHY: px covers most charts in one line; go unlocks dashboards and fine-grained control when you need it.

# Step 1: Import necessary libraries
import pandas as pd
import numpy as np
import plotly.express as px  # High-level interactive plotting (easy)
import plotly.graph_objects as go  # Low-level interactive plotting (flexible)
from plotly.subplots import make_subplots  # Multiple interactive plots

print("✅ Libraries imported successfully!")
print("\n📚 What each library does:")
print("   - pandas: Data manipulation (DataFrames)")
print("   - numpy: Numerical operations")
print("   - plotly.express: Easy interactive plots (like Seaborn for interactive)")
print("   - plotly.graph_objects: Advanced interactive plots (full control)")
print("   - plotly.subplots: Multiple interactive plots in one figure")

print("\n" + "=" * 70)
print("Example 4: Interactive Plotly Visualizations")
print("=" * 70)
print("\n📚 Prerequisites: Examples 02-03 completed, static visualization knowledge")
print("🔗 This is Example 04 in Unit 3 - interactive visualizations")
print("🎯 Goal: Master interactive plots for data exploration and dashboards\n")

✅ Libraries imported successfully!

📚 What each library does:
   - pandas: Data manipulation (DataFrames)
   - numpy: Numerical operations
   - plotly.express: Easy interactive plots (like Seaborn for interactive)
   - plotly.graph_objects: Advanced interactive plots (full control)
   - plotly.subplots: Multiple interactive plots in one figure

Example 4: Interactive Plotly Visualizations

📚 Prerequisites: Examples 02-03 completed, static visualization knowledge
🔗 This is Example 04 in Unit 3 - interactive visualizations
🎯 Goal: Master interactive plots for data exploration and dashboards



# 1. LOAD THE REAL 911 CALL LOG

In [2]:
# WHAT: Load the real Montgomery County 911 dispatch log and derive the columns the charts need.
# WHY: Interactive charts earn their keep on data that has more columns than one chart can show -
#      hovering reveals the township, ZIP and exact call type behind every point.

print("\n1. Loading the Real 911 Call Log")
print("-" * 70)

DATA_DIR = '../../../Course 04/datasets/raw/'

# Read only the columns we chart - usecols keeps a 123 MB file fast to load.
calls = pd.read_csv(DATA_DIR + 'montgomery_911_calls.csv',
                    usecols=['lat', 'lng', 'zip', 'title', 'timeStamp', 'twp'],
                    parse_dates=['timeStamp'])

# 'title' looks like 'EMS: BACK PAINS/INJURY' - the part before ':' is the service category.
calls['category'] = calls['title'].str.split(':').str[0]
calls['reason'] = calls['title'].str.split(': ').str[1].str.strip()
calls['hour'] = calls['timeStamp'].dt.hour

print(f"✓ Loaded {len(calls):,} real 911 calls "
      f"({calls['timeStamp'].min():%Y-%m-%d} to {calls['timeStamp'].max():%Y-%m-%d})")
print(f"  Categories: {calls['category'].value_counts().to_dict()}")
print(f"  Missing ZIP codes (real gaps, nothing was removed): {calls['zip'].isna().sum():,}")

# Classroom sample: 2,000 calls keeps the interactive scatter responsive in a browser.
# random_state=42 makes the sample reproducible - randomness here is the SAMPLING, not the data.
df = calls.sample(n=2000, random_state=42).copy()

# Size encoding: how busy is the township this call came from? (a real, computed quantity)
twp_volume = calls['twp'].value_counts()
df['twp_call_volume'] = df['twp'].map(twp_volume).fillna(0)

print(f"\n✓ Working sample: {len(df):,} calls")
print(df[['timeStamp', 'twp', 'category', 'reason', 'lat', 'lng']].head())


1. Loading the Real 911 Call Log
----------------------------------------------------------------------


✓ Loaded 663,522 real 911 calls (2015-12-10 to 2020-07-29)
  Categories: {'EMS': 332692, 'Traffic': 230208, 'Fire': 100622}
  Missing ZIP codes (real gaps, nothing was removed): 80,199

✓ Working sample: 2,000 calls
                 timeStamp                twp category              reason  \
528196 2019-07-31 05:13:54      LOWER GWYNEDD      EMS   CARDIAC EMERGENCY   
537685 2019-08-23 13:32:45  WEST CONSHOHOCKEN  Traffic  VEHICLE ACCIDENT -   
304176 2018-02-02 04:08:55     UPPER MORELAND     Fire          FIRE ALARM   
474703 2019-03-22 04:33:46      UPPER HANOVER      EMS         FALL VICTIM   
583079 2019-12-11 17:15:46         CHELTENHAM  Traffic  DISABLED VEHICLE -   

              lat        lng  
528196  40.169545 -75.249764  
537685  40.069832 -75.316295  
304176  40.154698 -75.139243  
474703  40.382529 -75.473491  
583079  40.091609 -75.138451  


# 2. INTERACTIVE SCATTER PLOT


In [3]:
# WHAT: Make an interactive scatter of WHERE the calls happened, coloured by service, saved to HTML.
# WHY: The HTML file is self-contained - anyone with a browser can pan around the county, no Python needed.

print("\n2. Interactive Scatter Plot: where the calls came from")
print("-" * 70)
fig = px.scatter(df, x='lng', y='lat', color='category',
                 size='twp_call_volume',
                 hover_data=['twp', 'reason', 'zip'],
                 title='Montgomery County 911 calls (2,000-call sample): incident locations',
                 labels={'lng': 'Longitude', 'lat': 'Latitude',
                         'category': 'Service', 'twp_call_volume': 'Calls from this township'})
fig.update_layout(
    font=dict(size=12), title_font_size=16,
    width=800,
    height=600
)
fig.write_html('11_interactive_scatter.html')
print("✓ Interactive scatter plot saved as HTML")
print("  Open '11_interactive_scatter.html' in a browser - hover any point to see")
print("  the township, the exact call type and the ZIP code behind it.")


2. Interactive Scatter Plot: where the calls came from
----------------------------------------------------------------------
✓ Interactive scatter plot saved as HTML
  Open '11_interactive_scatter.html' in a browser - hover any point to see
  the township, the exact call type and the ZIP code behind it.


# 3. INTERACTIVE LINE PLOT


In [4]:
# WHAT: Make an interactive monthly call-volume line chart with a range slider.
# WHY: Range sliders let readers zoom into any time window themselves - one chart serves every zoom level.

print("\n3. Interactive Line Plot: monthly call volume, 2015-2020")
print("-" * 70)

# Aggregate the FULL log (not the sample) to one point per month per service.
monthly = (calls.groupby([calls['timeStamp'].dt.to_period('M').dt.to_timestamp(), 'category'])
                .size().reset_index(name='calls')
                .rename(columns={'timeStamp': 'month'}))

fig = px.line(monthly, x='month', y='calls', color='category',
              title='911 calls per month by service (Montgomery County, PA)',
              labels={'month': 'Month', 'calls': 'Calls dispatched', 'category': 'Service'})
fig.update_traces(mode='lines+markers', marker_size=5)
fig.update_xaxes(rangeslider_visible=True)
fig.write_html('12_interactive_line.html')
print(f"✓ Interactive line plot saved with range slider ({len(monthly)} month-service points)")
print("  Note the first and last months are partial - the log starts 2015-12-10 and ends 2020-07-29.")


3. Interactive Line Plot: monthly call volume, 2015-2020
----------------------------------------------------------------------


✓ Interactive line plot saved with range slider (168 month-service points)
  Note the first and last months are partial - the log starts 2015-12-10 and ends 2020-07-29.


# 4. INTERACTIVE BAR CHART


In [5]:
# WHAT: Make an interactive bar chart of the ten most common call types, with value labels.
# WHY: Aggregating first (value_counts) then plotting keeps the chart honest and the tooltip exact.

print("\n4. Interactive Bar Chart: the ten most common reasons people call 911")
print("-" * 70)
top_reasons = (calls['reason'].value_counts().head(10)
               .rename_axis('reason').reset_index(name='calls'))
fig = px.bar(top_reasons, x='calls', y='reason', orientation='h',
             title='Ten most common 911 call reasons (all 663,522 calls)',
             labels={'reason': 'Call reason', 'calls': 'Number of calls'},
             color='calls', text='calls')
fig.update_traces(texttemplate='%{text:,}', textposition='outside')
fig.update_layout(showlegend=False, yaxis=dict(autorange='reversed'), width=900)
fig.write_html('13_interactive_bar.html')
print("✓ Interactive bar chart saved")
print(top_reasons.to_string(index=False))

# Also keep per-category counts - the dashboard below reuses them.
category_counts = (calls['category'].value_counts()
                   .rename_axis('category').reset_index(name='calls'))


4. Interactive Bar Chart: the ten most common reasons people call 911
----------------------------------------------------------------------


✓ Interactive bar chart saved
               reason  calls
   VEHICLE ACCIDENT - 148372
   DISABLED VEHICLE -  47909
           FIRE ALARM  38452
     VEHICLE ACCIDENT  36377
          FALL VICTIM  34683
RESPIRATORY EMERGENCY  34250
    CARDIAC EMERGENCY  32339
   ROAD OBSTRUCTION -  23235
      SUBJECT IN PAIN  19650
          HEAD INJURY  18304


# 5. MULTI-PANEL DASHBOARD


In [6]:
# WHAT: Assemble map-scatter, monthly trend, category bars, and hour-of-day histogram into one 2x2 dashboard.
# WHY: Dashboards put related views side by side so a reader can cross-check patterns on one screen.

print("\n5. Multi Panel Dashboard")
print("-" * 70)
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Call locations (sample)', 'Calls per month',
                    'Calls by service', 'Calls by hour of day'),
    specs=[[{"secondary_y": False}, {"secondary_y": False}],
           [{"secondary_y": False}, {"type": "histogram"}]]
)
# Scatter: where
fig.add_trace(
    go.Scatter(x=df['lng'], y=df['lat'], mode='markers',
               marker=dict(color=df['hour'], colorscale='Viridis', size=6,
                           colorbar=dict(title='Hour', x=0.46, len=0.42, y=0.79)),
               name='Calls', text=df['twp']),
    row=1, col=1
)
# Line: when (monthly, all calls)
total_monthly = monthly.groupby('month', as_index=False)['calls'].sum()
fig.add_trace(
    go.Scatter(x=total_monthly['month'], y=total_monthly['calls'],
               mode='lines+markers', name='Calls/month'),
    row=1, col=2
)
# Bar: what
fig.add_trace(
    go.Bar(x=category_counts['category'], y=category_counts['calls'], name='By service'),
    row=2, col=1
)
# Histogram: what hour
fig.add_trace(
    go.Histogram(x=calls['hour'], nbinsx=24, name='By hour'),
    row=2, col=2
)
fig.update_layout(
    height=800, title_text="Montgomery County 911 dispatch dashboard",
    showlegend=True
)
fig.write_html('14_interactive_dashboard.html')
print("✓ Interactive dashboard saved")

busiest_hour = int(calls['hour'].value_counts().idxmax())
print(f"  Busiest hour of the day in the real log: {busiest_hour:02d}:00 "
      f"({calls['hour'].value_counts().max():,} calls)")


5. Multi Panel Dashboard
----------------------------------------------------------------------
✓ Interactive dashboard saved
  Busiest hour of the day in the real log: 17:00 (44,119 calls)


# 6. 3D SCATTER PLOT


In [7]:
# WHAT: Make a 3D interactive scatter - longitude, latitude and hour of day.
# WHY: Rotation makes 3D readable - a spinnable cloud can reveal structure a flat map hides,
#      such as which parts of the county call at night.

print("\n6. 3D Scatter Plot: location plus time of day")
print("-" * 70)
fig = px.scatter_3d(df, x='lng', y='lat', z='hour', color='category',
                    size='twp_call_volume', hover_data=['twp', 'reason'],
                    title='911 calls in space and time (longitude, latitude, hour of day)',
                    labels={'lng': 'Longitude', 'lat': 'Latitude', 'hour': 'Hour of day'})
fig.write_html('15_3d_scatter.html')
print("✓ 3D scatter plot saved - drag to rotate, and look at the 02:00-05:00 layer")


6. 3D Scatter Plot: location plus time of day
----------------------------------------------------------------------


✓ 3D scatter plot saved - drag to rotate, and look at the 02:00-05:00 layer


# 7. SUMMARY


In [8]:
# WHAT: Print the notebook summary.
# WHY: The takeaway: Plotly writes standalone HTML - interactivity you can email or publish.

print("\n" + "=" * 70)
print("Summary")
print("=" * 70)
print(f"""
Everything above was drawn from {len(calls):,} REAL 911 calls
(Montgomery County, PA, {calls['timeStamp'].min():%b %Y} - {calls['timeStamp'].max():%b %Y}).

Key Concepts Covered:
1. Interactive scatter plots with hover information (township, reason, ZIP)
2. Interactive line plots with range sliders (monthly call volume)
3. Interactive bar charts (the ten most common call reasons)
4. Multi-panel dashboards (where / when / what / what hour)
5. 3D visualizations (longitude, latitude, hour of day)

Note: Plotly HTML files can be opened in any web browser.
Next Steps: Continue to Example 05 for more Plotly practice,
then Example 06 for customizing and annotating charts.
""")


Summary

Everything above was drawn from 663,522 REAL 911 calls
(Montgomery County, PA, Dec 2015 - Jul 2020).

Key Concepts Covered:
1. Interactive scatter plots with hover information (township, reason, ZIP)
2. Interactive line plots with range sliders (monthly call volume)
3. Interactive bar charts (the ten most common call reasons)
4. Multi-panel dashboards (where / when / what / what hour)
5. 3D visualizations (longitude, latitude, hour of day)

Note: Plotly HTML files can be opened in any web browser.
Next Steps: Continue to Example 05 for more Plotly practice,
then Example 06 for customizing and annotating charts.



## 📚 References

1. Bostock, M., Ogievetsky, V., & Heer, J. (2011). *D3: Data-Driven Documents*. IEEE Transactions on Visualization and Computer Graphics, 17(12), 2301-2309. <https://doi.org/10.1109/TVCG.2011.185>
2. Heer, J., & Shneiderman, B. (2012). *Interactive Dynamics for Visual Analysis*. ACM Queue, 10(2), 30-55. <https://doi.org/10.1145/2133416.2146416>
3. Wilkinson, L. (2005). *The Grammar of Graphics*, 2nd ed. Springer.